# SAPLMA

Fit a three-hidden-layer classifier to token activations near the end of the decoder. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2304.13734)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

The estimator trains for 3 epochs by default. Each epoch reports token-weighted training BCE, validation BCE, and validation token PR-AUC and ROC-AUC. Held-out metrics use the final epoch.

In [1]:
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import logging

import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS, PILOT_LIMITS
from src.evaluation import TraceStore, evaluate
from src.methods import SAPLMA

logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("src.methods").setLevel(logging.INFO)

epochs = SAPLMA.default_epochs
pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
history = []
for benchmark in DATASETS:
    print(f"Training on {benchmark}", flush=True)
    splits = store.load_splits(benchmark, limits)
    estimator = SAPLMA(epochs=epochs)
    results.extend(
        {"benchmark": benchmark, **row} for row in evaluate(estimator, splits)
    )
    history.extend(
        {"benchmark": benchmark, **row} for row in estimator.training_history
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

Training on mbpp


SAPLMA epoch 1/3 train_loss=0.4012 validation_loss=0.3106 validation_token_pr_auc=0.5656 validation_token_roc_auc=0.9000


SAPLMA epoch 2/3 train_loss=0.2199 validation_loss=0.2734 validation_token_pr_auc=0.6214 validation_token_roc_auc=0.9200


SAPLMA epoch 3/3 train_loss=0.1527 validation_loss=0.3017 validation_token_pr_auc=0.6291 validation_token_roc_auc=0.9188


Training on humaneval


SAPLMA epoch 1/3 train_loss=0.3904 validation_loss=0.6342 validation_token_pr_auc=0.3338 validation_token_roc_auc=0.5996


... (7 outputs left out)

SAPLMA epoch 1/3 train_loss=0.4373 validation_loss=0.5356 validation_token_pr_auc=0.7127 validation_token_roc_auc=0.8139


SAPLMA epoch 2/3 train_loss=0.2969 validation_loss=0.5713 validation_token_pr_auc=0.7116 validation_token_roc_auc=0.8144


SAPLMA epoch 3/3 train_loss=0.2396 validation_loss=0.6496 validation_token_pr_auc=0.6971 validation_token_roc_auc=0.8133


n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.7093   
               line     509         54         0.1061  0.7598   
               sample    74         31         0.4189  0.8984   
humaneval      token   3714        665         0.1791  0.4836   
               line     484         55         0.1136  0.5987   
               sample    46         19         0.4130  0.6818   
humaneval_java token   1157        205         0.1772  0.2724   
               line     155         15         0.0968  0.2739   
               sample    36         11         0.3056  0.3369   
defects4j      token   3901       2220         0.5691  0.9262   
               line     448        148         0.3304  0.8625   
               sample    86         35         0.4070  0.6138   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.7097   0.9172     0.6301  0.7227  0.6733   
               line               0.7630   0.9678     0.6308  0.7593  0.6891   
               sample             0.9002   0.9265     0.6596  1.0000  0.7949   
humaneval      token              0.4841   0.8090     0.2130  1.0000  0.3512   
               line               0.6025   0.9121     0.1698  1.0000  0.2902   
               sample             0.6913   0.7018     0.5000  0.7895  0.6122   
humaneval_java token              0.2751   0.6855     0.2670  0.5561  0.3608   
               line               0.2957   0.8681     0.2500  0.8667  0.3881   
               sample             0.3728   0.6073     0.3750  0.8182  0.5143   
defects4j      token              0.9262   0.9014     0.7190  0.9509  0.8189   
               line               0.8631   0.9298     0.6524  0.9257  0.7654   
               sample             0.6202   0.6448     0.4198  0.9714  0.5862   

                       threshold  
benchmark      level              
mbpp           token      0.4475  
               line       0.9165  
               sample     0.8950  
humaneval      token      0.0001  
               line       0.0130  
               sample     0.5107  
humaneval_java token      0.1375  
               line       0.2287  
               sample     0.4958  
defects4j      token      0.1256  
               line       0.7626  
               sample     0.6955

### Training history

Loss is binary cross-entropy on labeled tokens.

In [4]:
with pd.option_context("display.max_rows", None):
    display(pd.DataFrame(history).set_index(["benchmark", "epoch"]).round(4))

train_loss  validation_loss  validation_token_pr_auc  \
benchmark      epoch                                                         
mbpp           1          0.4012           0.3106                   0.5656   
               2          0.2199           0.2734                   0.6214   
               3          0.1527           0.3017                   0.6291   
humaneval      1          0.3904           0.6342                   0.3338   
               2          0.2759           0.9459                   0.3479   
               3          0.2115           1.1334                   0.3268   
humaneval_java 1          0.4791           0.3342                   0.2025   
               2          0.3725           0.3481                   0.2151   
               3          0.3182           0.3781                   0.1746   
defects4j      1          0.4373           0.5356                   0.7127   
               2          0.2969           0.5713                   0.7116   
               3          0.2396           0.6496                   0.6971   

                      validation_token_roc_auc  
benchmark      epoch                            
mbpp           1                        0.9000  
               2                        0.9200  
               3                        0.9188  
humaneval      1                        0.5996  
               2                        0.5786  
               3                        0.5723  
humaneval_java 1                        0.6829  
               2                        0.6944  
               3                        0.6755  
defects4j      1                        0.8139  
               2                        0.8144  
               3                        0.8133